In [1]:
import numpy as np
import bacco

In [6]:
import os
os.chdir("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [7]:
name_list = ['LH_{:d}'.format(i) for i in range(30)] + ['fiducial']

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

snap = 264
zoom = {}

loaded = []
for i in range(len(name_list)):
    if i<30:
        base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/LH_{:d}/hydro_output/".format(i)
    else:
        base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/fiducial/hydro_output/"
    zoom[name_list[i]] = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                            tau=tau, ns=ns, sigma8=sigma8, dm_file="snapdir_{:03d}/snapshot_{:03d}".format(snap,snap), use_ids=True, numpart=4320)


2026-02-17 23:42:06,998 bacco.sims : Initialising simulation Default
2026-02-17 23:42:06,999 bacco.sims : try /cosmos_storage/simulations/TNG_Family/MN5_resims/LH_0/hydro_output/snapdir_264/snapshot_264
2026-02-17 23:42:07,001 bacco.sims : Loading /cosmos_storage/simulations/TNG_Family/MN5_resims/LH_0/hydro_output/snapdir_264/snapshot_264
2026-02-17 23:42:07,454 bacco.sims : ...done in 0.00807 s
2026-02-17 23:42:07,455 bacco.sims : Initialising simulation Default
2026-02-17 23:42:07,455 bacco.sims : try /cosmos_storage/simulations/TNG_Family/MN5_resims/LH_1/hydro_output/snapdir_264/snapshot_264
2026-02-17 23:42:07,457 bacco.sims : Loading /cosmos_storage/simulations/TNG_Family/MN5_resims/LH_1/hydro_output/snapdir_264/snapshot_264
2026-02-17 23:42:07,873 bacco.sims : ...done in 0.00471 s
2026-02-17 23:42:07,873 bacco.sims : Initialising simulation Default
2026-02-17 23:42:07,874 bacco.sims : try /cosmos_storage/simulations/TNG_Family/MN5_resims/LH_2/hydro_output/snapdir_264/snapshot_264

In [8]:
for i, key in enumerate(zoom['fiducial'].sub.keys()): print(i, key)

0 nsubs_in_file
1 pos
2 vel
3 mostboundID
4 len
5 subnr
6 fof_index
7 central
8 vmax
9 rmax
10 vpeak
11 peak_time
12 mpeak
13 vel_core
14 id
15 id_offsets
16 star_id_offsets
17 mdot_at_mpeak
18 mdot
19 orphan_snap
20 infall_snap
21 subhalo_moment_of_inertia
22 subhalo_stellar_MOI
23 subhalo_red_moi
24 star_red_moi
25 subhalo_core_moi
26 star_core_moi
27 subhalo_bound_moi
28 star_bound_moi
29 subhalo_bound_moi_25
30 star_bound_moi_25
31 subhalo_bound_moi_75
32 star_bound_moi_75
33 subhalo_bound_moi_95
34 star_bound_moi_95
35 subhalo_bound_moi_90
36 star_bound_moi_90
37 subhalo_bound_moi_85
38 star_bound_moi_85
39 subhalo_bound_moi_80
40 star_bound_moi_80
41 bias
42 BHMdot
43 StarMetalFractions
44 WindMass
45 GasMetallicityMaxRad
46 VelDisp
47 SFR
48 StarMetallicityMaxRad
49 SFRinHalfRad
50 StarMetalFractionsHalfRad
51 GasMetalFractionsMaxRad
52 GasMetallicity
53 BHMass
54 IDMostbound
55 HalfmassRad
56 Parent
57 Spin
58 GasMetalFractions
59 StarMetallicityHalfRad
60 LenType
61 GasMetalli

In [9]:
# Load the Halo Selection
with open("/cosmos_storage/simulations/TNG_Family/MN5_resims/resims_info/hydro_halo_sel_1pmbin.txt") as f:
    final_sel = []

    for line in f.readlines():
        final_sel.append(int(line.split()[0]))

final_sel = np.array(final_sel)

# Perform the cross-match with MTNG halos
xmatch = {}

for i in range(len(name_list)):
    xmatch[name_list[i]] = utils.cross_match(zoom[name_list[i]], snap=264, name=name_list[i])

# Load MTNG and get the fraction of halos to do the upweighting
mtng = bacco.utils.load_MTNG(adr="/cosmos_storage/simulations/TNG_Family/MTNG/", snap=264)

m200b = np.log10(1e10 * mtng.fof['halo_m200b'])

mbins = np.concatenate(
    (np.arange(11, 11.5, 0.0025),
    np.arange(11.5, 12.5, 0.005),
    np.arange(12.5, 13.5, 0.025),
    np.arange(13.5, 15.01, 0.125))
)

h_frac = np.zeros(len(final_sel))
for m in range(len(mbins)-1):
    h_frac[m] = np.where(( m200b[final_sel]>=mbins[m]) & ( m200b[final_sel]<mbins[m+1]))[0].shape[0] / \
             np.where(( m200b>=mbins[m]) & ( m200b<mbins[m+1]))[0].shape[0]

zoom_split = {}
zoom_sel = {}
for i in range(len(name_list)):
    zoom_split[name_list[i]] = utils.split_halos(zoom[name_list[i]])

    zoom_sel[name_list[i]] = {}

    zoom_sel[name_list[i]]['sel'] = xmatch[name_list[i]]['ind'][:,np.newaxis,np.newaxis]
    zoom_sel[name_list[i]]['h_frac'] = h_frac[np.newaxis, :]

2026-02-17 23:42:13,896 bacco.sims : Initialising simulation Default
2026-02-17 23:42:13,897 bacco.sims : try /cosmos_storage/simulations/TNG_Family/MTNG/groups_264/fof_subhalo_tab_264
2026-02-17 23:42:13,898 bacco.sims : Loading /cosmos_storage/simulations/TNG_Family/MTNG/groups_264/fof_subhalo_tab_264
2026-02-17 23:42:13,926 bacco.sims : ...done in 0.00491 s
2026-02-17 23:42:13,929 bacco.sims : Reading 98233510 items for Group_M_Mean200


/tmp/ipykernel_38346/87356854.py:19: RuntimeWarning: divide by zero encountered in log10
  m200b = np.log10(1e10 * mtng.fof['halo_m200b'])


In [14]:
zoom_sel['fiducial']['sel'].shape

(452, 1, 1)

In [15]:
zoom_split['fiducial'].rhalf_m2half(sel_mask=zoom_sel['fiducial'], nbins=20)

IndexError: too many indices for array: array is 1-dimensional, but 3 were indexed

In [68]:
def SFR_mstar(self, sel_mask=None, nbins=100):

    # I don't really want this as a function of mass, but instead of redshift I believe
    bins = np.logspace(8, 13, nbins)

    first = self.sim.fof['halo_firstsub']
    nsubs = self.sim.fof['halo_nsubs']
    
    counts     = np.zeros(nbins-1)
    weights    = np.zeros(nbins-1)
    sSFR_mean   = np.zeros(nbins-1)
    mstar_mean = np.zeros(nbins-1)

    for m in range(len(sel_mask['sel'])):
        if sel_mask['h_frac'][0][m]!=0:

            mstar = []
            sSFR  = []
            for i in range(len(sel_mask['sel'][m][0])):
                # Get all the subhalos inside of a certain halo in the selection
                mstar.extend(self.sim.sub['MassType'][:,4][first[sel_mask['sel'][m][0][i]]:first[sel_mask['sel'][m][0][i]]+nsubs[sel_mask['sel'][m][0][i]]])
                sSFR.extend(self.sim.sub['SFR'][first[sel_mask['sel'][m][0][i]]:first[sel_mask['sel'][m][0][i]]+nsubs[sel_mask['sel'][m][0][i]]])

            mstar = 1e10 * np.array(mstar)
            sSFR  = 1e10 * np.array(sSFR) / mstar

            ids = np.digitize(mstar, bins)

            counts_i = np.array([np.sum( np.ones(len(mstar))[np.where(ids==j)]) for j in range(1,len(bins))])
            weights += counts_i / sel_mask['h_frac'][0][m]
            counts  += counts_i

            sSFR_mean += np.array([np.sum(sSFR[np.where(ids==j)]) for j in range(1,len(bins))]) / sel_mask['h_frac'][0][m]
            mstar_mean += np.array([np.sum(mstar[np.where(ids==j)]) for j in range(1,len(bins))])

    sSFR_mean /= weights
    mstar_mean /= counts

    return {'sSFR':sSFR_mean, 'mstar':mstar_mean, 'counts':counts}

In [ ]:
mhalo_edges = np.logspace(11, 14.5, )